In [1]:
import pandas as pd
import numpy as np

In [2]:
input_path = "../../results/choice_model/input.parquet"
df = pd.read_parquet(input_path)

modes = ["car", "car_passenger", "transit", "bicycle", "walk"]

for column in df.columns:
    if column.startswith("motorbike_"):
        modes.append("motorbike")
        break

In [4]:
# remove
before = len(df)
df = df[df["mode"].isin(modes)]
after = len(df)
print(f"Removed {before - after} rows, not in modes")

Removed 89 rows, not in modes


In [5]:
for mode in modes:
    df["{}_availability".format(mode)] = df["{}_availability".format(mode)].astype(int)

df["mode"] = df["mode"].apply(modes.index)
df["weight"] = len(df) * df["weight"] / df["weight"].sum()

df = df.drop(columns = ["origin_iris", "destination_iris"])

for column in ["has_driving_permit", "has_motorbike_permit", "has_pt_subscription", "transit_only_bus"]:
    if column in df:
        df[column] = df[column].astype(int)

In [6]:
import biogeme.database as db
import biogeme.biogeme as bio
from biogeme import models
from biogeme.expressions import Beta, Variable, bioNormalCdf, Elem, log

In [7]:
database = db.Database("data", df)

mode = db.Variable("mode")
weight = db.Variable("weight")
euclidean_distance_km = db.Variable("euclidean_distance_km")

# person
person_has_driving_permit = db.Variable("has_driving_permit")

# income
if "income_per_person_EUR" in df:
    income_per_person_EUR = db.Variable("income_EUR")

# car
car_availability = db.Variable("car_availability")
car_in_vehicle_time_min = db.Variable("car_in_vehicle_time_min")
car_walk_time_min = db.Variable("car_walk_time_min")
car_cost_EUR = db.Variable("car_cost_EUR")

# parking
parking_cost_EUR = db.Variable("parking_cost_EUR")
parking_pressure = db.Variable("parking_pressure")

# car passenger
car_passenger_availability = db.Variable("car_passenger_availability")
car_passenger_in_vehicle_time_min = db.Variable("car_passenger_in_vehicle_time_min")
car_passenger_walk_time_min = db.Variable("car_passenger_walk_time_min")


if "motorbike" in modes:
    # motorbike
    motorbike_availability = db.Variable("motorbike_availability")
    motorbike_in_vehicle_time_min = db.Variable("motorbike_in_vehicle_time_min")
    motorbike_walk_time_min = db.Variable("motorbike_walk_time_min")
    motorbike_cost_EUR = db.Variable("motorbike_cost_EUR")

# transit
transit_availability = db.Variable("transit_availability")
transit_total_walk_time_min = db.Variable("transit_total_walk_time_min")
transit_total_in_vehicle_time_min = db.Variable("transit_total_in_vehicle_time_min")
transit_transfers = db.Variable("transit_transfers")
transit_transfer_wait_time_min = db.Variable("transit_transfer_wait_time_min")
transit_initial_wait_time_min = db.Variable("transit_initial_wait_time_min")
transit_cost_EUR = db.Variable("transit_cost_EUR")
transit_in_vehicle_time_bus_min = db.Variable("transit_in_vehicle_time_bus_min")
transit_only_bus = db.Variable("transit_only_bus")

# bicycle
bicycle_availability = db.Variable("bicycle_availability")
bicycle_travel_time_min = db.Variable("bicycle_travel_time_min")

# walk
walk_availability = db.Variable("walk_availability")
walk_travel_time_min = db.Variable("walk_travel_time_min")




In [8]:
lambda_cost_distance = Beta("lambda_cost_distance", -0.1, None, None, 0)
lambda_cost_income = Beta("lambda_cost_income", -0.1, None, None, 0)

beta_cost_EUR = Beta("beta_cost_EUR", 0, None, None, 0)

beta_car_asc = Beta("beta_car_asc", 0, None, None, 0)
beta_car_in_vehicle_time_min = Beta("beta_car_in_vehicle_time_min", 0, None, None, 0)
beta_car_walk_time_min = Beta("beta_car_walk_time_min", 0, None, None, 0)
beta_car_parking_pressure = Beta("beta_car_parking_pressure", 0, None, None, 0)

beta_car_passenger_asc = Beta("beta_car_passenger_asc", 0, None, None, 0)
beta_car_passenger_in_vehicle_time_min = Beta("beta_car_passenger_in_vehicle_time_min", 0, None, None, 0)
beta_car_passenger_walk_time_min = Beta("beta_car_passenger_walk_time_min", 0, None, None, 0)
beta_car_passenger_parking_pressure = Beta("beta_car_passenger_parking_pressure", 0, None, None, 0)
beta_car_passenger_driving_permit = Beta("beta_car_passenger_driving_permit", 0, None, None, 0)

if "motorbike" in modes:
    beta_motorbike_asc = Beta("beta_motorbike_asc", 0, None, None, 0)
    beta_motorbike_in_vehicle_time_min = Beta("beta_motorbike_in_vehicle_time_min", 0, None, None, 0)
    beta_motorbike_walk_time_min = Beta("beta_motorbike_walk_time_min", 0, None, None, 0)

beta_transit_asc = Beta("beta_transit_asc", 0, None, None, 1)
beta_transit_total_walk_time_min = Beta("beta_transit_total_walk_time_min", 0, None, None, 0)
beta_transit_headway_min = Beta("beta_transit_headway_min", 0, None, None, 0)
beta_transit_total_in_vehicle_time_min = Beta("beta_transit_in_vehicle_time_total_min", 0, None, None, 0)
beta_transit_transfers = Beta("beta_transit_transfers", 0, None, None, 0)
beta_transit_waiting_time_min = Beta("beta_transit_waiting_time_min", 0, None, None, 0)
beta_transit_in_vehicle_time_bus_min = Beta("beta_transit_in_vehicle_time_bus_min", 0, None, None, 0)
beta_transit_only_bus = Beta("beta_transit_only_bus", 0, None, None, 0)
beta_transit_driving_permit = Beta("beta_transit_driving_permit", 0, None, None, 0)

beta_bicycle_asc = Beta("beta_bicycle_asc", 0, None, None, 0)
beta_bicycle_travel_time_min = Beta("beta_bicycle_travel_time_min", 0, None, None, 0)

beta_walk_asc = Beta("beta_walk_asc", 0, None, None, 0)
beta_walk_travel_time_min = Beta("beta_walk_travel_time_min", 0, None, None, 0)

beta_access_time_min = Beta("beta_access_time_min", 0, None, None, 0)
beta_car_walk_time_min = beta_access_time_min
beta_car_passenger_walk_time_min =  beta_access_time_min
beta_motorbike_walk_time_min = beta_access_time_min
beta_transit_total_walk_time_min =  beta_access_time_min

In [10]:
# Utility functions

mean_euclidean_distance_km = 4.4
euclidean_interaction_cost = (euclidean_distance_km / mean_euclidean_distance_km)**lambda_cost_distance
# euclidean_interaction_cost = 1

if "income_per_person_EUR" in df:
    mean_income_EUR = 2900
    income_interaction_cost = (income_per_person_EUR / mean_income_EUR)**lambda_cost_income
else:
    income_interaction_cost = 1


lambda_cost_distance = Beta("lambda_cost_distance", -0.1, None, None, 0)
lambda_cost_income = Beta("lambda_cost_income", -0.1, None, None, 0)

beta_cost_EUR_C1 = Beta("beta_cost_EUR", 0, None, None, 0)
beta_cost_EUR_C2 = Beta("beta_cost_EUR", 0, None, None, 0)

beta_car_asc_C1 = Beta("beta_car_asc", 0, None, None, 0)
beta_car_asc_C2 = Beta("beta_car_asc", 0, None, None, 0)

beta_car_in_vehicle_time_min_C1 = Beta("beta_car_in_vehicle_time_min", 0, None, None, 0)
beta_car_in_vehicle_time_min_C2 = Beta("beta_car_in_vehicle_time_min", 0, None, None, 0)


beta_car_passenger_asc_C1 = Beta("beta_car_passenger_asc", 0, None, None, 0)
beta_car_passenger_asc_C2 = Beta("beta_car_passenger_asc", 0, None, None, 0)

beta_car_passenger_in_vehicle_time_min_C1 = Beta("beta_car_passenger_in_vehicle_time_min", 0, None, None, 0)
beta_car_passenger_in_vehicle_time_min_C2 = Beta("beta_car_passenger_in_vehicle_time_min", 0, None, None, 0)

if "motorbike" in modes:
    beta_motorbike_asc_C1 = Beta("beta_motorbike_asc", 0, None, None, 0)
    beta_motorbike_asc_C2 = Beta("beta_motorbike_asc", 0, None, None, 0)
    beta_motorbike_in_vehicle_time_min_C1 = Beta("beta_motorbike_in_vehicle_time_min", 0, None, None, 0)
    beta_motorbike_in_vehicle_time_min_C2 = Beta("beta_motorbike_in_vehicle_time_min", 0, None, None, 0)

beta_transit_asc_C1 = Beta("beta_transit_asc", 0, None, None, 1)
beta_transit_asc_C2 = Beta("beta_transit_asc", 0, None, None, 1)


beta_transit_total_in_vehicle_time_min_C1 = Beta("beta_transit_in_vehicle_time_total_min", 0, None, None, 0)
beta_transit_total_in_vehicle_time_min_C2 = Beta("beta_transit_in_vehicle_time_total_min", 0, None, None, 0)


beta_bicycle_asc_C1 = Beta("beta_bicycle_asc", 0, None, None, 0)
beta_bicycle_asc_C2 = Beta("beta_bicycle_asc", 0, None, None, 0)


beta_bicycle_travel_time_min_C1 = Beta("beta_bicycle_travel_time_min", 0, None, None, 0)
beta_bicycle_travel_time_min_C2 = Beta("beta_bicycle_travel_time_min", 0, None, None, 0)

beta_walk_asc_C1 = Beta("beta_walk_asc", 0, None, None, 0)
beta_walk_asc_C2 = Beta("beta_walk_asc", 0, None, None, 0)

beta_walk_travel_time_min_C1 = Beta("beta_walk_travel_time_min", 0, None, None, 0)
beta_walk_travel_time_min_C2 = Beta("beta_walk_travel_time_min", 0, None, None, 0)


availability = { 
    modes.index("car"): car_availability,
    modes.index("car_passenger"): car_passenger_availability,
    modes.index("transit"): transit_availability,
    modes.index("bicycle"): bicycle_availability,
    modes.index("walk"): walk_availability
}

if "motorbike" in modes:
    availability[modes.index("motorbike")] = motorbike_availability

# Probabilités d'appartenance aux classes latentes
SCALE_CLASS1 = Beta('SCALE_CLASS1', 0, None, None, 0)
SCALE_CLASS2 = Beta('SCALE_CLASS2', 0, None, None, 1)

P_CLASS1 = models.logit({1: SCALE_CLASS1, 2: SCALE_CLASS2}, availability, 1)
P_CLASS2 = 1 - P_CLASS1

# Utility functions

mean_euclidean_distance_km = 4.4
euclidean_interaction_cost = (euclidean_distance_km / mean_euclidean_distance_km)**lambda_cost_distance
# euclidean_interaction_cost = 1

if "income_per_person_EUR" in df:
    mean_income_EUR = 2900
    income_interaction_cost = (income_per_person_EUR / mean_income_EUR)**lambda_cost_income
else:
    income_interaction_cost = 1

car_utility_C1 = beta_car_asc_C1 + beta_car_in_vehicle_time_min_C1 * car_in_vehicle_time_min + beta_cost_EUR_C1 * car_cost_EUR  * euclidean_interaction_cost 
car_utility_C2 = beta_car_asc_C2 + beta_car_in_vehicle_time_min_C2 * car_in_vehicle_time_min + beta_cost_EUR_C2 * car_cost_EUR  * euclidean_interaction_cost 



car_passenger_utility_C1 = beta_car_passenger_asc_C1 + beta_car_passenger_in_vehicle_time_min_C1 * car_passenger_in_vehicle_time_min
car_passenger_utility_C2 = beta_car_passenger_asc_C2 + beta_car_passenger_in_vehicle_time_min_C2 * car_passenger_in_vehicle_time_min

if "motorbike" in modes:
    motorbike_utility_C1 = beta_motorbike_asc_C1  + beta_motorbike_in_vehicle_time_min_C1 * motorbike_in_vehicle_time_min  + beta_cost_EUR_C1 * motorbike_cost_EUR * euclidean_interaction_cost 
    motorbike_utility_C2 = beta_motorbike_asc_C2  + beta_motorbike_in_vehicle_time_min_C2 * motorbike_in_vehicle_time_min + beta_cost_EUR_C2 * motorbike_cost_EUR * euclidean_interaction_cost 

transit_utility_C1 = beta_transit_asc_C1 + beta_transit_total_in_vehicle_time_min_C1 * transit_total_in_vehicle_time_min + beta_cost_EUR_C1 * transit_cost_EUR * euclidean_interaction_cost 
transit_utility_C2 = beta_transit_asc_C2 + beta_transit_total_in_vehicle_time_min_C2 * transit_total_in_vehicle_time_min  + beta_cost_EUR_C2 * transit_cost_EUR * euclidean_interaction_cost 

bicycle_utility_C1 = beta_bicycle_asc_C1 +  beta_bicycle_travel_time_min_C1 * bicycle_travel_time_min
bicycle_utility_C2 = beta_bicycle_asc_C2 +  beta_bicycle_travel_time_min_C2 * bicycle_travel_time_min

walk_utility_C1 = beta_walk_asc_C1 + beta_walk_travel_time_min_C1 * walk_travel_time_min
walk_utility_C2 = beta_walk_asc_C2 + beta_walk_travel_time_min_C2 * walk_travel_time_min


# Mapping
utilities_C1 = { 
    modes.index("car"): car_utility_C1,
    modes.index("car_passenger"): car_passenger_utility_C1,
    modes.index("transit"): transit_utility_C1,
    modes.index("bicycle"): bicycle_utility_C1,
    modes.index("walk"): walk_utility_C1
}

if "motorbike" in modes:
    utilities_C1[modes.index("motorbike")] = motorbike_utility_C1

    # Mapping
utilities_C2 = { 
    modes.index("car"): car_utility_C2,
    modes.index("car_passenger"): car_passenger_utility_C2,
    modes.index("transit"): transit_utility_C2,
    modes.index("bicycle"): bicycle_utility_C2,
    modes.index("walk"): walk_utility_C2
}

if "motorbike" in modes:
    utilities_C2[modes.index("motorbike")] = motorbike_utility_C2


# Modèle logit conditionnel pour chaque classe
P_MODE_C1 = models.logit(utilities_C1, mode)
P_MODE_C2 = models.logit(utilities_C2, mode)

# Modèle final de classes latentes (pondération des deux classes)
P = P_CLASS1 * P_MODE_C1 + P_CLASS2 * P_MODE_C2

loglikelihood = weight * log(P)
biogeme = bio.BIOGEME(database, loglikelihood)
biogeme.modelName = "latent_class_model_full"

results = biogeme.estimate()

# Affichage des résultats
print(results.getEstimatedParameters())

TypeError: logit() missing 1 required positional argument: 'i'